# 01 — Classificatore a feature (TF-IDF + regressione logistica)

Sul dataset definitivo (272 testate). Due input:

- **A** — solo il testo dell'articolo;
- **C** — il testo dell'articolo + il testo delle pagine del sito della sua testata, in coda (le stesse pagine
  per ogni articolo della testata).

Fold **per gruppo** (principale: nessuna testata sorella nel training) e **per testata** (confronto).
Un modello per fold: vocabolario e IDF si stimano **solo sul training** del fold, il test passa solo da
`transform`. Iperparametri fissi, nessuna scelta guarda il test.

Salva, per ogni input e schema, `Results/2-classificazione/feature_<A|C>_<schema>_articoli.csv` con le
probabilità per classe `p_l1..p_l5`, `y_pred`, `y_true`, `domain`, `fold`; e le letture in
`feature_metriche.csv`. Se un file c'è già, lo ricarica senza riaddestrare.
Log: `Results/2-classificazione/avanzamento_feature.log`.

In [ ]:
import time

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

from comune import CLASSI, NOME_SCHEMA, SCHEMI, SEED, carica, cartella_risultati, letture, registratore

In [ ]:
CONFIG = ["A", "C"]        # A = solo articoli, C = articoli + pagine
SCHEMI_DA_FARE = SCHEMI    # fold per gruppo e per testata
FOLDS = None               # quali fold (None = tutti)
PROVA = False              # True: 40 testate, risultati in Results/2-classificazione/prova/

In [ ]:
OUT = cartella_risultati(PROVA)
registra = registratore(OUT / "avanzamento_feature.log")
art = carica(PROVA)
art["A"] = art.testo
art["C"] = art.testo + "\n\n" + art.pagine
registra(f"=== dataset definitivo: {len(art)} articoli, {art.domain.nunique()} testate ===")

## Addestramento

Per ogni fold: `fit_transform` sul training, `transform` sul test. Le parole del test mai viste in
addestramento non hanno colonna e vengono ignorate.

In [ ]:
def addestra(config, schema):
    file_out = OUT / f"feature_{config}_{schema}_articoli.csv"
    if file_out.exists():
        registra(f"{config} {schema}: già fatto, carico {file_out.name}")
        return pd.read_csv(file_out)
    proba = np.zeros((len(art), 5))
    folds = FOLDS if FOLDS else sorted(art[schema].unique())
    for fold in folds:
        t0 = time.time()
        tr, te = art[art[schema] != fold], art[art[schema] == fold]
        registra(f"{config} {schema} fold {fold}: inizio ({len(tr)} training, {len(te)} test)")
        v = TfidfVectorizer(ngram_range=(1, 2), min_df=5, max_df=0.9, sublinear_tf=True,
                            max_features=100_000, strip_accents="unicode")
        clf = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED)
        clf.fit(v.fit_transform(tr[config]), tr.label)       # vocabolario, IDF e modello: solo training
        p = clf.predict_proba(v.transform(te[config]))       # test: solo transform
        proba[te.index.to_numpy()[:, None], clf.classes_[None, :]] = p
        registra(f"{config} {schema} fold {fold}: fatto in {time.time()-t0:.0f} s, "
                 f"acc articoli {accuracy_score(te.label, p.argmax(1)):.3f}")
    pred = pd.DataFrame(proba, columns=CLASSI)
    pred.insert(0, "domain", art.domain.values)
    pred.insert(0, "article_id", art.article_id.values)
    pred["fold"] = art[schema].values
    pred["y_true"] = art.label.values
    pred["y_pred"] = proba.argmax(1)
    pred = pred[pred.fold.isin(folds)].reset_index(drop=True)   # solo i fold davvero eseguiti
    pred.to_csv(file_out, index=False)
    registra(f"=== {config} {schema}: salvato ({len(pred)} righe con probabilità) ===")
    return pred


pred = {(c, s): addestra(c, s) for c in CONFIG for s in SCHEMI_DA_FARE}

## Risultati

Per articolo e per testata (voto di maggioranza): accuratezza a 5 classi, F1 macro, binario (soglia tra
$l_2$ e $l_3$, punteggio NewsGuard 60) e tolerant ($\alpha$ = 0.5 e 1).

In [ ]:
ris = pd.DataFrame([{"config": c, "fold": NOME_SCHEMA[s], **letture(p)} for (c, s), p in pred.items()])
ris = ris.set_index(["config", "fold"])
ris.to_csv(OUT / "feature_metriche.csv")
ris